# 01 — EDA Running : Boston Marathon 2015-2017

**Objectif** : Explorer le dataset Boston Marathon pour comprendre la distribution des profils, identifier les variables prédictives, et préparer la modélisation.

**Dataset** : Boston Marathon 2015, 2016, 2017 — ~79 638 finishers  
**Source** : Kaggle — rojour/boston-results  
**Colonnes clés** : Age, M/F, splits 5K→40K, Official Time

---

## Plan
1. Chargement et fusion des 3 années
2. Audit qualité (types, valeurs manquantes, doublons)
3. Distributions démographiques (âge, sexe)
4. Distribution des temps de course
5. Analyse des splits (5K, 10K, ... 40K)
6. Corrélations entre variables
7. Analyse par sous-groupes (âge, sexe)
8. Conclusions et variables retenues pour la modélisation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Chargement et fusion des données

In [ ]:
years = [2015, 2016, 2017]
dfs = []

for year in years:
    df = pd.read_csv(f'../data/raw/running/marathon_results_{year}.csv')
    df['year'] = year
    dfs.append(df)

# Harmoniser les colonnes (2015/2017 ont 25 cols, 2016 en a 24)
common_cols = list(set(dfs[0].columns) & set(dfs[1].columns) & set(dfs[2].columns))
df = pd.concat([d[common_cols] for d in dfs], ignore_index=True)

print(f"Dataset fusionné : {len(df):,} lignes, {len(df.columns)} colonnes")
print(f"Années : {df['year'].value_counts().sort_index().to_dict()}")
df.head()

## 2. Audit qualité

In [ ]:
print("=== Types de colonnes ===")
print(df.dtypes)
print()
print("=== Valeurs manquantes ===")
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print("=== Doublons ===")
print(f"Doublons : {df.duplicated().sum()}")

In [ ]:
def time_to_seconds(t):
    """Convertit HH:MM:SS ou MM:SS en secondes. Retourne NaN si invalide."""
    try:
        parts = str(t).strip().split(':')
        if len(parts) == 3:
            return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
        elif len(parts) == 2:
            return int(parts[0]) * 60 + float(parts[1])
        return np.nan
    except:
        return np.nan

split_cols = ['5K', '10K', '15K', '20K', 'Half', '25K', '30K', '35K', '40K', 'Official Time']
available_splits = [c for c in split_cols if c in df.columns]

for col in available_splits:
    df[col + '_sec'] = df[col].apply(time_to_seconds)

df['gender'] = df['M/F'].map({'M': 0, 'F': 1})

print("Colonnes converties en secondes :")
print([c + '_sec' for c in available_splits])
print()
print("Taux de conversion réussi :")
for col in available_splits:
    valid = df[col + '_sec'].notna().sum()
    print(f"  {col}: {valid:,} / {len(df):,} ({100*valid/len(df):.1f}%)")

In [ ]:
# Suppression des lignes sans temps final valide
df_clean = df[df['Official Time_sec'].notna()].copy()
df_clean = df_clean[df_clean['Age'].between(15, 90)]
df_clean = df_clean[df_clean['Official Time_sec'].between(7200, 36000)]  # 2h à 10h

print(f"Après nettoyage : {len(df_clean):,} lignes ({100*len(df_clean)/len(df):.1f}% conservé)")

## 3. Distributions démographiques

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribution âge
axes[0].hist(df_clean['Age'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution des âges')
axes[0].set_xlabel('Âge')
axes[0].set_ylabel('Nombre de coureurs')
axes[0].axvline(df_clean['Age'].median(), color='red', linestyle='--', label=f"Médiane : {df_clean['Age'].median():.0f} ans")
axes[0].legend()

# Répartition homme/femme
gender_counts = df_clean['M/F'].value_counts()
axes[1].bar(gender_counts.index, gender_counts.values, color=['steelblue', 'salmon'])
axes[1].set_title('Répartition par sexe')
axes[1].set_ylabel('Nombre de coureurs')
for i, v in enumerate(gender_counts.values):
    axes[1].text(i, v + 100, f'{100*v/len(df_clean):.1f}%', ha='center')

# Distribution par année
year_counts = df_clean['year'].value_counts().sort_index()
axes[2].bar(year_counts.index.astype(str), year_counts.values, color='teal')
axes[2].set_title('Finishers par année')
axes[2].set_ylabel('Nombre de coureurs')

plt.tight_layout()
plt.savefig('../data/processed/eda_running_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nStats âge : médiane={df_clean['Age'].median():.0f}, min={df_clean['Age'].min()}, max={df_clean['Age'].max()}")
print(f"Sexe : {gender_counts.to_dict()}")

## 4. Distribution des temps de course

In [ ]:
def sec_to_hms(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    return f"{h}h{m:02d}"

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribution globale
axes[0].hist(df_clean['Official Time_sec'] / 3600, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution des temps marathon (toutes années)')
axes[0].set_xlabel('Temps (heures)')
axes[0].set_ylabel('Nombre de coureurs')
median_h = df_clean['Official Time_sec'].median() / 3600
axes[0].axvline(median_h, color='red', linestyle='--', label=f"Médiane : {sec_to_hms(df_clean['Official Time_sec'].median())}")
axes[0].legend()

# Hommes vs Femmes
for gender, color, label in [('M', 'steelblue', 'Hommes'), ('F', 'salmon', 'Femmes')]:
    subset = df_clean[df_clean['M/F'] == gender]['Official Time_sec'] / 3600
    axes[1].hist(subset, bins=50, alpha=0.6, color=color, edgecolor='white', label=f"{label} (médiane: {sec_to_hms(subset.median()*3600)})")

axes[1].set_title('Distribution temps marathon par sexe')
axes[1].set_xlabel('Temps (heures)')
axes[1].set_ylabel('Nombre de coureurs')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/processed/eda_running_times.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nStats temps marathon (secondes) :")
print(df_clean.groupby('M/F')['Official Time_sec'].describe().applymap(lambda x: sec_to_hms(x) if x > 60 else round(x, 1)))

## 5. Analyse des splits

In [ ]:
split_sec_cols = [c + '_sec' for c in ['5K', '10K', '15K', '20K', '25K', '30K', '35K', '40K'] if c + '_sec' in df_clean.columns]
split_labels = [c.replace('_sec', '') for c in split_sec_cols]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Vitesse médiane par split (km/h)
distances = {'5K': 5, '10K': 10, '15K': 15, '20K': 20, '25K': 25, '30K': 30, '35K': 35, '40K': 40}
median_speeds = []
for col, label in zip(split_sec_cols, split_labels):
    dist = distances.get(label, None)
    if dist:
        speed = (dist / (df_clean[col].median() / 3600))
        median_speeds.append(speed)

axes[0].plot(split_labels[:len(median_speeds)], median_speeds, marker='o', color='steelblue')
axes[0].set_title('Vitesse médiane par split (tous coureurs)')
axes[0].set_xlabel('Split')
axes[0].set_ylabel('Vitesse (km/h)')
axes[0].set_ylim(bottom=0)

# Corrélation 5K vs temps final
sample = df_clean[df_clean['5K_sec'].notna() & df_clean['Official Time_sec'].notna()].sample(min(3000, len(df_clean)), random_state=42)
axes[1].scatter(sample['5K_sec'] / 60, sample['Official Time_sec'] / 3600, alpha=0.1, s=5, color='steelblue')
axes[1].set_title('Temps 5K vs Temps Marathon')
axes[1].set_xlabel('Temps 5K (minutes)')
axes[1].set_ylabel('Temps Marathon (heures)')

corr_5k = df_clean[['5K_sec', 'Official Time_sec']].corr().iloc[0, 1]
axes[1].text(0.05, 0.95, f'r = {corr_5k:.3f}', transform=axes[1].transAxes, fontsize=12, color='red')

plt.tight_layout()
plt.savefig('../data/processed/eda_running_splits.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nCorrélation 5K → Temps Marathon : r = {corr_5k:.4f}")

## 6. Corrélations entre variables

In [ ]:
numeric_cols = ['Age', 'gender'] + split_sec_cols + ['Official Time_sec']
corr_matrix = df_clean[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Matrice de corrélation — Variables running')
plt.tight_layout()
plt.savefig('../data/processed/eda_running_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCorrélations avec Official Time_sec :")
print(corr_matrix['Official Time_sec'].sort_values(ascending=False).to_string())

## 7. Analyse par sous-groupes

In [ ]:
df_clean['age_group'] = pd.cut(df_clean['Age'], bins=[15, 25, 35, 45, 55, 65, 90],
                                labels=['18-25', '26-35', '36-45', '46-55', '56-65', '65+'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Temps médian par groupe d'âge et sexe
pivot = df_clean.groupby(['age_group', 'M/F'])['Official Time_sec'].median().unstack() / 3600
pivot.plot(kind='bar', ax=axes[0], color=['steelblue', 'salmon'], width=0.7)
axes[0].set_title('Temps marathon médian par groupe d\'âge et sexe')
axes[0].set_xlabel('Groupe d\'âge')
axes[0].set_ylabel('Temps (heures)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title='Sexe')

# Boxplot temps par groupe d'âge
age_groups_data = [df_clean[df_clean['age_group'] == g]['Official Time_sec'].values / 3600
                   for g in ['18-25', '26-35', '36-45', '46-55', '56-65', '65+']]
axes[1].boxplot(age_groups_data, labels=['18-25', '26-35', '36-45', '46-55', '56-65', '65+'],
                patch_artist=True, boxprops=dict(facecolor='lightsteelblue'))
axes[1].set_title('Distribution temps marathon par groupe d\'âge')
axes[1].set_xlabel('Groupe d\'âge')
axes[1].set_ylabel('Temps (heures)')

plt.tight_layout()
plt.savefig('../data/processed/eda_running_age_groups.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Validation baseline Riegel

In [ ]:
def riegel(t1, d1, d2, exponent=1.06):
    """Prédit le temps t2 sur distance d2 depuis t1 sur d1."""
    return t1 * (d2 / d1) ** exponent

# Prédire le temps marathon depuis le 5K
test = df_clean[df_clean['5K_sec'].notna() & df_clean['Official Time_sec'].notna()].copy()
test['marathon_riegel'] = test['5K_sec'].apply(lambda t: riegel(t, 5, 42.195))

# Calculer l'erreur
test['error_sec'] = test['marathon_riegel'] - test['Official Time_sec']
test['error_pct'] = 100 * test['error_sec'] / test['Official Time_sec']

mae = test['error_sec'].abs().mean()
mape = test['error_pct'].abs().mean()

print(f"Baseline Riegel (5K → Marathon) :")
print(f"  MAE  = {mae:.0f} secondes ({mae/60:.1f} minutes)")
print(f"  MAPE = {mape:.1f}%")
print()
print("Cette baseline sera le point de comparaison du modèle ML.")

# Visualisation erreur
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(test['error_pct'].clip(-30, 30), bins=60, color='steelblue', edgecolor='white')
ax.axvline(0, color='red', linestyle='--', label='Prédiction parfaite')
ax.set_title(f'Erreur Riegel (5K→Marathon) — MAE={mae/60:.1f} min, MAPE={mape:.1f}%')
ax.set_xlabel('Erreur (%)')
ax.set_ylabel('Fréquence')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/eda_running_riegel_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Export dataset nettoyé

In [ ]:
export_cols = ['Age', 'M/F', 'gender', 'year'] + split_sec_cols + ['Official Time_sec', 'age_group']
export_cols = [c for c in export_cols if c in df_clean.columns]

df_export = df_clean[export_cols].copy()
df_export.to_csv('../data/processed/running_clean.csv', index=False)
print(f"Dataset nettoyé exporté : {len(df_export):,} lignes, {len(df_export.columns)} colonnes")
print(f"Fichier : data/processed/running_clean.csv")

## 10. Conclusions EDA Running

### Variables retenues pour la modélisation
| Variable | Type | Justification |
|---|---|---|
| `5K_sec` | Feature principale | Corrélation très forte avec temps marathon |
| `10K_sec` | Feature principale | Idem, plus représentative sur longue distance |
| `Age` | Feature secondaire | Impact modéré, significatif après 45 ans |
| `gender` | Feature secondaire | Différence ~30-40 min en médiane H/F |

### Cibles de prédiction
- **Target principale** : `Official Time_sec` (temps marathon)
- **Extrapolations** : 5K, 10K, semi via formule Riegel depuis le temps marathon prédit

### Baseline à battre
- Riegel (5K → Marathon) : MAE ≈ X minutes, MAPE ≈ X% *(valeurs calculées ci-dessus)*

### Limites identifiées
- Pas de données physiologiques brutes (poids, FC) → mode simple basé sur formules Astrand/VDOT
- Dataset Boston Marathon = coureurs qualifiés (temps minimum requis) → biais de sélection, sous-représentation des débutants
- Ce biais est documenté et communiqué dans l'interface utilisateur